# 08 - Linear Algebra in Action (Capstone)

This notebook ties everything together. We will build a quantum circuit from scratch using only linear algebra, then verify that our results match what Qiskit produces.

The circuit: Create a Bell state using a Hadamard gate followed by a CNOT gate.

In [ ]:
import numpy as np

## Step 1: Define Our Qubit States

We start with two qubits, both in state |0>. The combined state is |00>.

In [ ]:
# Single qubit basis
ket_0 = np.array([[1], [0]], dtype=complex)
ket_1 = np.array([[0], [1]], dtype=complex)

# Initial 2-qubit state: |00>
initial_state = np.kron(ket_0, ket_0)
print("Initial state |00>:")
print(initial_state.flatten())

## Step 2: Define Our Gates as Matrices

In [ ]:
# Hadamard gate (2x2)
H = np.array([[1, 1],
              [1, -1]], dtype=complex) / np.sqrt(2)

# Identity gate (2x2)
I = np.eye(2, dtype=complex)

# CNOT gate (4x4) -- flips target qubit if control qubit is |1>
CNOT = np.array([[1, 0, 0, 0],
                 [0, 1, 0, 0],
                 [0, 0, 0, 1],
                 [0, 0, 1, 0]], dtype=complex)

print("Hadamard:")
print(H.round(4))
print("\nCNOT:")
print(CNOT)

## Step 3: Apply Hadamard to Qubit 1 Only

To apply H to qubit 1 and do nothing to qubit 2, we compute H tensor I.

In [ ]:
# H on qubit 1, I on qubit 2
H_I = np.kron(H, I)

# Apply to |00>
after_hadamard = H_I @ initial_state
print("After H on qubit 1:")
print(after_hadamard.flatten().round(4))

# This should be (|00> + |10>) / sqrt(2)
# = [1/sqrt(2), 0, 1/sqrt(2), 0]

print("\nProbabilities:")
for i, label in enumerate(['00', '01', '10', '11']):
    print(f"  P(|{label}>) = {abs(after_hadamard[i, 0])**2:.4f}")

## Step 4: Apply CNOT

CNOT flips qubit 2 if qubit 1 is |1>. This creates entanglement.

In [ ]:
# Apply CNOT
after_cnot = CNOT @ after_hadamard
print("After CNOT (Bell state):")
print(after_cnot.flatten().round(4))

# This should be (|00> + |11>) / sqrt(2)
# = [1/sqrt(2), 0, 0, 1/sqrt(2)]

print("\nProbabilities:")
for i, label in enumerate(['00', '01', '10', '11']):
    print(f"  P(|{label}>) = {abs(after_cnot[i, 0])**2:.4f}")

print("\n50% |00>, 50% |11> -- this is the Bell state. Qubits are entangled.")

## Step 5: Combined Circuit as One Matrix

The entire circuit (H then CNOT) can be written as a single 4x4 matrix.

In [ ]:
# Combined circuit matrix
circuit = CNOT @ H_I  # CNOT applied after H

print("Full circuit matrix:")
print(circuit.round(4))

# Apply to |00> in one step
result = circuit @ initial_state
print("\nResult:")
print(result.flatten().round(4))

# Verify it matches our step-by-step result
print(f"\nMatches step-by-step? {np.allclose(result, after_cnot)}")

# Verify the circuit matrix is unitary
print(f"Circuit is unitary? {np.allclose(circuit @ circuit.conj().T, np.eye(4))}")

## Step 6: Measurement Probabilities Using Inner Products

The probability of measuring a specific state is |<basis_state|final_state>|^2.

In [ ]:
# Build all 2-qubit basis states
basis_states = {
    '00': np.kron(ket_0, ket_0),
    '01': np.kron(ket_0, ket_1),
    '10': np.kron(ket_1, ket_0),
    '11': np.kron(ket_1, ket_1),
}

bell_state = result  # Our Bell state from above

print("Measurement probabilities using inner products:")
total = 0
for label, basis in basis_states.items():
    amplitude = basis.conj().T @ bell_state  # <basis|bell>
    prob = abs(amplitude[0, 0])**2
    total += prob
    print(f"  |<{label}|Bell>|^2 = {prob:.4f}")
print(f"  Total = {total:.4f}")

## Step 7: Verify with Qiskit

Now let us build the same circuit in Qiskit and check that our math produces the same result.

If you do not have Qiskit installed, run: `pip install qiskit`

In [ ]:
try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector

    # Build the Bell circuit in Qiskit
    qc = QuantumCircuit(2)
    qc.h(0)      # Hadamard on qubit 0
    qc.cx(0, 1)  # CNOT with control=0, target=1

    print("Qiskit circuit:")
    print(qc.draw())

    # Get the statevector
    sv = Statevector.from_instruction(qc)
    print("\nQiskit statevector:", sv.data.round(4))

    # Our result
    print("Our result:        ", result.flatten().round(4))

    # Note: Qiskit uses little-endian qubit ordering
    # So the order might be different but the probabilities match
    print("\nQiskit probabilities:")
    probs = sv.probabilities_dict()
    for state, prob in sorted(probs.items()):
        print(f"  P(|{state}>) = {prob:.4f}")

except ImportError:
    print("Qiskit is not installed.")
    print("Install it with: pip install qiskit")
    print("\nBut our linear algebra result is correct regardless:")
    print("Bell state =", result.flatten().round(4))
    print("50% |00>, 50% |11>")

## What We Just Did (Summary)

We built a quantum circuit entirely from linear algebra:

1. **Vectors** -- represented qubit states as column vectors
2. **Tensor products** -- combined two qubits into a 4D state space
3. **Matrices** -- represented quantum gates (H, CNOT)
4. **Matrix multiplication** -- applied gates to states
5. **Inner products** -- computed measurement probabilities
6. **Unitarity** -- verified the circuit preserves probability

Every concept from notebooks 01-07 was used here. This is the math behind every quantum circuit.

## Final Exercises

1. Build a 3-qubit GHZ state: (|000> + |111>) / sqrt(2). Use H on qubit 1, then CNOT(1,2), then CNOT(1,3). Verify the probabilities.

2. Build a circuit that applies X to qubit 1, then H to qubit 1, then measure. What are the probabilities? Work it out with matrices first, then verify.

3. Take the Bell state and apply H tensor H (Hadamard to both qubits). What state do you get? What are the new probabilities?

In [ ]:
# Your code here
